In [0]:
from pyspark.sql import functions as F
import re

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.gold")

# Create checkpoint table to track processed files
spark.sql("""
CREATE TABLE IF NOT EXISTS company_ro.bronze.metadata_files_processed (
  file_path STRING,
  table_name STRING,
  ingested_at TIMESTAMP,
  row_count BIGINT,
  file_size BIGINT
)
USING DELTA
""")

print("✓ Checkpoint table ready")

In [0]:
bucket = "s3://ro-company-lake"
onrc_firme_path = f"{bucket}/raw_v2/onrc/firme/"

display(dbutils.fs.ls(onrc_firme_path))

In [0]:
# List all available snapshot dates
all_snapshots = dbutils.fs.ls(onrc_firme_path)
snapshot_dates = [
    re.search(r"snapshot_date=(\d{4}-\d{2}-\d{2})", s.path).group(1)
    for s in all_snapshots
    if "snapshot_date=" in s.path
]

print(f"Available snapshots: {sorted(snapshot_dates)}")

# Get already-processed snapshot dates from checkpoint (serverless-compatible)
processed_rows = (
    spark.table("company_ro.bronze.metadata_files_processed")
    .filter(F.col("table_name") == "onrc_firme_raw")
    .select("file_path")
    .collect()
)
processed_snapshots = [row["file_path"] for row in processed_rows]

processed_dates = [
    re.search(r"snapshot_date=(\d{4}-\d{2}-\d{2})", path).group(1)
    for path in processed_snapshots
    if "snapshot_date=" in path
]

print(f"Already processed: {sorted(set(processed_dates))}")

# Filter to only NEW snapshot dates
new_snapshots = [d for d in snapshot_dates if d not in processed_dates]

if not new_snapshots:
    print("\n✓ No new snapshots to process. All data already ingested.")
    dbutils.notebook.exit("No new data")

print(f"\nNew snapshots to process: {sorted(new_snapshots)}")

# Load ONLY new snapshots
od_firme = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("sep", "^")
    .option("encoding", "UTF-8")
    .csv([
        f"{onrc_firme_path}snapshot_date={date}/package=*/od_firme.csv"
        for date in new_snapshots
    ])
)

display(od_firme.limit(20))
row_count = od_firme.count()
print("New rows:", row_count)
print("Columns:", od_firme.columns)

In [0]:
# Append new data (not overwrite)
od_firme_with_metadata = (
    od_firme
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

(
    od_firme_with_metadata
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("company_ro.bronze.onrc_firme_raw")
)

print(f"✓ Appended {row_count:,} rows to company_ro.bronze.onrc_firme_raw")

# Record processed snapshots in checkpoint
checkpoint_records = []
for date in new_snapshots:
    path = f"{onrc_firme_path}snapshot_date={date}/package=*/od_firme.csv"
    checkpoint_records.append((path, "onrc_firme_raw", row_count // len(new_snapshots), 0))

checkpoint_df = (
    spark.createDataFrame(
        [(path, table_name, rows, size)
         for path, table_name, rows, size in checkpoint_records],
        ["file_path", "table_name", "row_count", "file_size"]
    )
    .withColumn("ingested_at", F.current_timestamp())
)

(
    checkpoint_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("company_ro.bronze.metadata_files_processed")
)

print(f"✓ Recorded {len(new_snapshots)} new snapshots in checkpoint table")

In [0]:
display(spark.sql("""
SELECT COUNT(*) AS rows
FROM company_ro.bronze.onrc_firme_raw
"""))